# Thalika — One‑Click Colab Studio

**နေ့စဉ်အသုံးပြုပုံ:** T4 GPU ချိတ် → **Runtime ▸ Run all** → Thalika app ပေါ်လာသည်အထိ စောင့်ပါ။

ဒီ Notebook က အောက်ပါတို့ကို အလိုအလျောက်လုပ်ပေးပါတယ်။

- GitHub ကနေ Thalika ကို download/update လုပ်ခြင်း
- Google Drive ထဲမှာ VoxCPM2 model cache သိမ်းခြင်း
- Node/Python dependencies တင်ခြင်း
- Colab T4, Matplotlib နဲ့ VoxCPM seed compatibility fixes ထည့်ခြင်း
- VoxCPM2 model server နဲ့ production Thalika app စတင်ခြင်း
- Colab iframe ထဲမှာ Thalika ကိုဖွင့်ခြင်း

> **Voice consent:** ကိုယ်ပိုင်အသံ သို့မဟုတ် ပိုင်ရှင်က ခွင့်ပြုထားသောအသံကိုသာ clone လုပ်ပါ။

<div style="padding:12px 16px;border:1px solid #E6E5E3;border-radius:10px;background:#F9F8F7">
<b>Colab limitation:</b> GPU ကို အမြဲတမ်းမရနိုင်ပါ။ “Failed to assign a backend” ပြရင် active sessions တွေပိတ်ပြီး ခဏစောင့်ကာ T4 ကိုပြန်ချိတ်ပါ။
</div>


In [ ]:
# 1/4 — GPU, persistent model cache, and GitHub project
from google.colab import drive
from pathlib import Path
import os
import shutil
import signal
import subprocess
import sys

REPO_URL = "https://github.com/htunlinn55287-cpu/thalika-colab.git"
PROJECT = Path("/content/Thalika")
USE_GOOGLE_DRIVE_CACHE = True

# Stop an older run before updating files. This makes Run all safe to repeat.
for process_name in ("app_process", "model_process"):
    process = globals().get(process_name)
    if process and process.poll() is None:
        try:
            os.killpg(os.getpgid(process.pid), signal.SIGTERM)
        except ProcessLookupError:
            pass

subprocess.run(
    "fuser -k 3000/tcp 7860/tcp >/dev/null 2>&1 || true",
    shell=True,
    check=False,
)

gpu = subprocess.run(
    ["nvidia-smi"],
    capture_output=True,
    text=True,
)
if gpu.returncode != 0:
    raise RuntimeError(
        "GPU မရသေးပါ။ Runtime > Change runtime type > T4 GPU ကိုရွေးပြီး ပြန်ချိတ်ပါ။"
    )
print(gpu.stdout)

if USE_GOOGLE_DRIVE_CACHE:
    drive.mount("/content/drive", force_remount=False)
    hf_home = Path("/content/drive/MyDrive/ThalikaCache/huggingface")
else:
    hf_home = Path("/content/huggingface")
hf_home.mkdir(parents=True, exist_ok=True)

os.environ.update(
    {
        "HF_HOME": str(hf_home),
        "HUGGINGFACE_HUB_CACHE": str(hf_home / "hub"),
        "HF_HUB_DOWNLOAD_TIMEOUT": "120",
        "HF_XET_HIGH_PERFORMANCE": "1",
        "HF_HUB_ENABLE_HF_TRANSFER": "1",
        "NEXT_TELEMETRY_DISABLED": "1",
        "MPLBACKEND": "Agg",
        "TORCHDYNAMO_DISABLE": "1",
        "TORCHINDUCTOR_DISABLE": "1",
        "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True",
        "VOXCPM_DEVICE": "cuda",
        "VOXCPM_TIMESTEPS": "10",
        "VOXCPM_LOAD_DENOISER": "0",
        "VOXCPM_SEED": "42",
    }
)

if (PROJECT / ".git").exists():
    print("Updating Thalika from GitHub...")
    subprocess.run(["git", "-C", str(PROJECT), "reset", "--hard"], check=True)
    subprocess.run(["git", "-C", str(PROJECT), "pull", "--ff-only"], check=True)
else:
    shutil.rmtree(PROJECT, ignore_errors=True)
    print("Downloading Thalika from GitHub...")
    subprocess.run(
        ["git", "clone", "--depth", "1", REPO_URL, str(PROJECT)],
        check=True,
    )

required = [
    PROJECT / "package.json",
    PROJECT / "src",
    PROJECT / "local-server" / "server.py",
    PROJECT / "local-server" / "requirements.txt",
]
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise RuntimeError(f"Incomplete GitHub project: {missing}")

os.chdir(PROJECT)
print("\nProject ready:", PROJECT)
print("Persistent model cache:", hf_home)


In [ ]:
# 2/4 — Install dependencies and apply Colab/T4 compatibility fixes
from pathlib import Path
import os
import re
import shutil
import subprocess
import sys

def node_major() -> int:
    if not shutil.which("node"):
        return 0
    result = subprocess.run(
        ["node", "--version"],
        capture_output=True,
        text=True,
    )
    try:
        return int(result.stdout.strip().lstrip("v").split(".")[0])
    except Exception:
        return 0

if node_major() < 22:
    subprocess.run(
        "curl -fsSL https://deb.nodesource.com/setup_22.x | bash -",
        shell=True,
        check=True,
    )
    subprocess.run(["apt-get", "install", "-y", "nodejs"], check=True)

subprocess.run(
    ["apt-get", "update", "-qq"],
    check=True,
)
subprocess.run(
    ["apt-get", "install", "-y", "-qq", "ffmpeg", "libsndfile1"],
    check=True,
)

print("Node:", subprocess.check_output(["node", "--version"], text=True).strip())
subprocess.run(["npm", "ci", "--no-audit", "--no-fund"], cwd=PROJECT, check=True)

# Colab Python 3.12 can fail while creating venv with ensurepip.
# virtualenv is reliable on Colab, so create the VoxCPM environment with it.
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--upgrade", "virtualenv"],
    check=True,
)
VENV = PROJECT / "local-server" / ".voxcpm-venv"
VENV_PYTHON = VENV / "bin" / "python"

if not VENV_PYTHON.exists():
    shutil.rmtree(VENV, ignore_errors=True)
    subprocess.run(
        [sys.executable, "-m", "virtualenv", str(VENV)],
        check=True,
    )

subprocess.run(
    [str(VENV_PYTHON), "-m", "pip", "install", "-q", "--upgrade", "pip"],
    check=True,
)
subprocess.run(
    [
        str(VENV_PYTHON),
        "-m",
        "pip",
        "install",
        "-r",
        str(PROJECT / "local-server" / "requirements.txt"),
    ],
    check=True,
)

server_file = PROJECT / "local-server" / "server.py"
server_text = server_file.read_text(encoding="utf-8")

# Set a headless Matplotlib backend and disable torch.compile on Tesla T4.
compile_marker = "# THALIKA_COLAB_T4_AND_MATPLOTLIB_FIX"
if compile_marker not in server_text:
    import_line = "import os\n"
    if import_line not in server_text:
        raise RuntimeError("Could not patch server.py: import os not found.")
    compatibility = """import os

# THALIKA_COLAB_T4_AND_MATPLOTLIB_FIX
os.environ.setdefault("MPLBACKEND", "Agg")
os.environ.setdefault("TORCHDYNAMO_DISABLE", "1")
os.environ.setdefault("TORCHINDUCTOR_DISABLE", "1")

import torch
if torch.cuda.is_available():
    capability = torch.cuda.get_device_capability(0)
    print(f"[thalika-local] CUDA capability: {capability[0]}.{capability[1]}")
    if capability[0] < 8:
print("[thalika-local] Tesla T4 detected — disabling torch.compile.")
torch.compile = lambda model, *args, **kwargs: model
"""
    server_text = server_text.replace(import_line, compatibility, 1)

# VoxCPM 2.0.3 does not accept seed= in model.generate().
server_text = re.sub(
    r'^[ \t]*"seed":[ \t]*seed_value,[ \t]*\n',
    "",
    server_text,
    flags=re.MULTILINE,
)

seed_marker = "# THALIKA_VOXCPM203_SEED_FIX"
if seed_marker not in server_text:
    pattern = re.compile(
        r'(?m)^(?P<i>[ \t]*)with model_lock:\n(?P=i)    wav = model\.generate\(\*\*kwargs\)'
    )
    match = pattern.search(server_text)
    if not match:
        raise RuntimeError("Could not patch server.py: generate block not found.")
    indent = match.group("i")
    replacement = (
        f"{indent}# THALIKA_VOXCPM203_SEED_FIX\n"
        f"{indent}import random\n\n"
        f"{indent}with model_lock:\n"
        f"{indent}    random.seed(seed_value)\n"
        f"{indent}    np.random.seed(seed_value)\n"
        f"{indent}    torch.manual_seed(seed_value)\n"
        f"{indent}    if torch.cuda.is_available():\n"
        f"{indent}        torch.cuda.manual_seed_all(seed_value)\n"
        f"{indent}    wav = model.generate(**kwargs)"
    )
    server_text = pattern.sub(replacement, server_text, count=1)

server_file.write_text(server_text, encoding="utf-8")
subprocess.run([str(VENV_PYTHON), "-m", "py_compile", str(server_file)], check=True)

(PROJECT / ".env.local").write_text(
    """HF_VOXCPM2_URL=http://127.0.0.1:7860
HF_REQUEST_TIMEOUT=120000
HF_INFERENCE_TIMEOUT=900000
VOXCPM_DEVICE=cuda
VOXCPM_TIMESTEPS=10
VOXCPM_LOAD_DENOISER=0
""",
    encoding="utf-8",
)

print("Dependencies and compatibility fixes are ready.")


In [ ]:
# 3/4 — Download/load VoxCPM2 and start the model server
from pathlib import Path
import os
import signal
import subprocess
import time
import requests

MODEL_LOG = Path("/content/thalika-voxcpm.log")
VENV_PYTHON = PROJECT / "local-server" / ".voxcpm-venv" / "bin" / "python"

old_model = globals().get("model_process")
if old_model and old_model.poll() is None:
    os.killpg(os.getpgid(old_model.pid), signal.SIGTERM)
    time.sleep(2)

model_env = os.environ.copy()
print("Checking/downloading VoxCPM2 model (~8 GB on the first run only)...")
subprocess.run(
    [
        str(VENV_PYTHON),
        "-c",
        (
            "from huggingface_hub import snapshot_download; "
            "snapshot_download('openbmb/VoxCPM2')"
        ),
    ],
    cwd=PROJECT / "local-server",
    env=model_env,
    check=True,
)

model_log_handle = MODEL_LOG.open("w")
model_process = subprocess.Popen(
    [str(VENV_PYTHON), "-u", "server.py"],
    cwd=PROJECT / "local-server",
    env=model_env,
    stdout=model_log_handle,
    stderr=subprocess.STDOUT,
    start_new_session=True,
)
print("Model PID:", model_process.pid)

deadline = time.time() + 45 * 60
last_notice = 0.0
while time.time() < deadline:
    if model_process.poll() is not None:
        print(MODEL_LOG.read_text(errors="replace")[-12000:])
        raise RuntimeError("VoxCPM2 stopped before becoming ready.")
    try:
        response = requests.get(
            "http://127.0.0.1:7860/gradio_api/info",
            timeout=5,
        )
        if response.ok:
            print("VoxCPM2 ready at http://127.0.0.1:7860")
            break
    except requests.RequestException:
        pass

    if time.time() - last_notice > 30:
        if MODEL_LOG.exists():
            print(MODEL_LOG.read_text(errors="replace")[-1000:])
        last_notice = time.time()
    time.sleep(5)
else:
    print(MODEL_LOG.read_text(errors="replace")[-12000:])
    raise TimeoutError("Model startup exceeded 45 minutes.")


In [ ]:
# 4/4 — Build production Thalika, start it, and open the app
from google.colab import output
from pathlib import Path
import os
import signal
import subprocess
import time
import requests

APP_LOG = Path("/content/thalika-production.log")

old_app = globals().get("app_process")
if old_app and old_app.poll() is None:
    os.killpg(os.getpgid(old_app.pid), signal.SIGTERM)
    time.sleep(2)

print("Building Thalika production app...")
build = subprocess.run(
    ["npm", "run", "build"],
    cwd=PROJECT,
    env=os.environ.copy(),
    text=True,
)
if build.returncode != 0:
    raise RuntimeError("Thalika production build failed.")
print("Production build completed.")

app_log_handle = APP_LOG.open("w")
app_process = subprocess.Popen(
    [
        "npm",
        "run",
        "start",
        "--",
        "--hostname",
        "0.0.0.0",
        "--port",
        "3000",
    ],
    cwd=PROJECT,
    env=os.environ.copy(),
    stdout=app_log_handle,
    stderr=subprocess.STDOUT,
    start_new_session=True,
)
print("Production app PID:", app_process.pid)

for _ in range(120):
    if app_process.poll() is not None:
        print(APP_LOG.read_text(errors="replace")[-10000:])
        raise RuntimeError("Thalika production app stopped during startup.")
    try:
        response = requests.get(
            "http://127.0.0.1:3000/api/health",
            timeout=4,
        )
        if response.ok:
            print("Thalika production app is ready.")
            break
    except requests.RequestException:
        pass
    time.sleep(2)
else:
    print(APP_LOG.read_text(errors="replace")[-10000:])
    raise RuntimeError("Thalika production app did not become ready.")

print("\nUse short text and Quality steps 10 for the first test.")
output.serve_kernel_port_as_iframe(3000, height=1050)


## အကြံပြု Settings

| အသုံးပြုပုံ | Style | Emotion | Intensity | Steps |
|---|---|---:|---:|---:|
| ပထမဆုံးစမ်းသပ်ခြင်း | Professional | Neutral | 50% | 10 |
| Movie recap | Movie recap | Tense / Dramatic | 65–80% | 10–20 |
| Emotional story | Cinematic | Warm / Sad / Hopeful | 65–80% | 10–20 |

- ပထမဆုံးအကြိမ်မှာ စာတိုတစ်ပိုဒ်နဲ့ စမ်းပါ။
- T4 မှာ Steps `10 → 16 → 20` လို တဖြည်းဖြည်းမြှင့်ပါ။
- Reference voice ကို ရှင်းလင်းသော 10–30 စက္ကန့် audio အသုံးပြုပါ။
- App iframe ကို မတော်တဆပိတ်မိရင် အောက်က **Reopen app** cell ကိုသာ Run ပါ။


In [ ]:
# Reopen app / optional diagnostics — this cell is safe during Run all
from google.colab import output
from pathlib import Path
import requests

if requests.get("http://127.0.0.1:3000/api/health", timeout=10).ok:
    output.serve_kernel_port_as_iframe(3000, height=1050)
else:
    print("Thalika app is not running. Rerun cell 4/4.")

SHOW_LOGS = False
if SHOW_LOGS:
    for filename in (
        "/content/thalika-voxcpm.log",
        "/content/Thalika/data/logs/generation.log",
        "/content/thalika-production.log",
    ):
        path = Path(filename)
        print("\n" + "=" * 24)
        print(filename)
        print("=" * 24)
        print(path.read_text(errors="replace")[-12000:] if path.exists() else "No log yet.")


### ပြဿနာဖြစ်ရင်

1. အပေါ်က diagnostics cell ထဲက `SHOW_LOGS = True` ပြောင်းပြီး Run ပါ။
2. Model server နဲ့ app ကို မပိတ်ဘဲ error ရဲ့ နောက်ဆုံးပိုင်းကိုစစ်ပါ။
3. Notebook အဆုံးမှာ shutdown cell မပါပါ—ဒါကြောင့် **Run all** ပြီးတာနဲ့ server မပိတ်သွားပါဘူး။
